# Process GPI TDM H+T by City

Computes city-level median household income and total households from ACS block-group
data, then uses median income to normalise H+T (Housing + Transportation) costs into an
income-share metric.  `HPLUST{YYYY}` = `(HTCOST{YYYY} × 12) / MEDINC{YYYY}` (0–1
annual share).  `HTCOST{YYYY}` is the monthly combined H+T cost.  `MEDINC{YYYY}`,
`TOTPOP{YYYY}`, and `TOTHH{YYYY}` (total households) are also exported.

Block-group ACS income suppressed by the Census Bureau is filled via a hierarchical
fallback — BG → parent census tract → parent county — before city-level aggregation.
Tract and county reference data are cached locally as CSV files and only re-downloaded
when the file does not already exist in `.\Inputs\`.

**To force a full re-download** of tract or county ACS data (e.g. after adding a new
target year), delete the corresponding CSV from `.\Inputs\` before running.

ACS income (city & county) is downloaded for **all ACS years 2009 – latest H+T year** (`acs_years`), while H+T cost columns drive `target_years` (H+T CSV years only).  The final export contains `MEDINC{yr}_CITY` and `MEDINC{yr}_CNTY` for every ACS year, with H+T cost/share columns only for years in the H+T CSV.

**Workflow**
1. Setup — imports, `fill_acs_panel` helper, config, output directories
2. Load Inputs — city boundaries, H+T CSV, derive target years
3. Build 2020 Block Group Reference — fetch via pygris, persist to GDB (skipped if exists)
4. Fetch ACS Block Group Data by Year — `get_census` per year/county (income, pop, HH)
5. Estimate Missing BG ACS Years — interpolate gaps; extrapolate edges via OLS trend
6. Fetch & Cache Tract-Level ACS Fallback — download once to CSV; apply same interp/extrap
7. Fetch & Cache County-Level ACS Fallback — same for county level
8. Apply Hierarchical Income Fallback (BG → Tract → County) — fill suppressed BG income
9. Assign Block Groups to Cities — arcpy centroid spatial join (`HAVE_THEIR_CENTER_IN`)
10. Aggregate to City-Level Income — household-weighted median income; sum of households
11. Fetch & Cache ACS Place Income / Build Hybrid MEDINC — Place > County > BG-weighted
12. Compute H+T Income Share and Build Export — HTCOST, HPLUST, MEDINC, TOTPOP, TOTHH
13. Validate Export

## 1. Setup

In [1]:
import arcpy
from arcpy import env
import os
import re
import numpy as np
import pandas as pd
from arcgis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import geopandas as gpd
from pygris import block_groups
from pygris.data import get_census
from shapely.geometry import MultiPolygon, Polygon

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"
pd.options.display.max_columns = None


In [2]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is not set in the ArcGIS Pro Python environment.")


In [3]:
# Output directories
outputs = [".\\Outputs", "scratch.gdb", "Affordability_Housing_Transportation_Costs.gdb"]

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])


In [4]:
# Config for the city-level ACS / block-group / income workflow.
# All sections in this notebook read from this dictionary.
HT_INCOME_CONFIG = {
    "ht_csv_path": r".\Inputs\H +T Costs for Dashboard 2019-2024 (GPI & TDM) - Composite H + T Metric.csv",
    # ── Block-group spatial reference (geometry, fetched once via pygris) ──────
    "bg_reference_gdb": r".\Inputs\block_groups_2020.gdb",
    "bg_reference_name": "bg_2020_ut_5county",
    "bg_projected_name": "bg_2020_projected",
    "bg_city_join_name": "bg_city_spatial_join",
    # ── Tract- & county-level ACS fallback (tabular, cached to CSV) ───────────
    # Naming mirrors block_groups_2020 so the geography level is immediately clear.
    # Delete these files to force a full re-download (e.g. after adding a new year).
    "tract_ref_path": r".\Inputs\tracts_2020_acs_income.csv",
    "county_ref_path": r".\Inputs\counties_2020_acs_income.csv",
    "places_ref_path": r".\Inputs\places_2020_acs_income.csv",  # ACS Place income cache
    # ── Geography identifiers ─────────────────────────────────────────────────
    "state": "UT",
    "state_fips": "49",
    "county_fips": ["003", "011", "035", "049", "057"],
    "county_names": ["BOX ELDER", "DAVIS", "SALT LAKE", "UTAH", "WEBER"],
    # ── ACS variables fetched at every geographic level ───────────────────────
    "acs_dataset": "acs/acs5",
    "acs_vars": {
        "median_income": "B19013_001E",  # Median household income
        "population": "B02001_001E",  # Total population (weight for income avg)
        "households": "B11001_001E",  # Total households (weight for H+T cost rollup)
    },
    # ── Column / key names ────────────────────────────────────────────────────
    "bg_key": "GEOID",
    "tract_key": "tract_geoid",  # derived as bg_geoid[:11]
    "county_key": "county_geoid",  # derived as bg_geoid[:5]
    # ── Notebook behaviour ────────────────────────────────────────────────────
    "year_prefixes": ["HCOST", "TCOST", "HPLUST"],
    "target_years_override": None,
    "pygris_cache": True,
    "share_scale": "0-1",
    "city_key": "CITYAREA",
    # ── ACS year range (independent of H+T CSV) ──────────────────────────────
    "acs_start_year": 2009,  # Earliest ACS 5-year API vintage
    # ── County name → FIPS lookup (matches "County" column in H+T CSV) ──────────
    "county_name_to_fips": {
        "Box Elder": "49003",
        "Davis": "49011",
        "Salt Lake": "49035",
        "Utah": "49049",
        "Weber": "49057",
    },
    "workshop_area_lookup": {
        "Box Elder (WFRC)": "Box Elder Wfrc",
        "North Davis County": "Davis County North",
        "South Davis County": "Davis County South",
        "North Salt Lake County": "Salt Lake County North",
        "Southwest Salt Lake County": "Salt Lake County Sw",
        "Southeast Salt Lake County": "Salt Lake County Se",
        "North Weber County": "Weber County North",
        "South Weber County": "Weber County South",
        "Central Utah County": "Utah County Central",
        "North Utah County": "Utah County North",
        "South Utah County": "Utah County South",
    },
}


In [5]:
def fill_acs_panel(panel_df, geoid_col, value_source_pairs, year_col="year"):
    """
    Apply linear interpolation (internal gaps) and OLS extrapolation (edge gaps)
    to a long-format ACS panel, processing each GEOID independently.

    Used for block-group, tract, and county panels — defined once here so the
    logic is not duplicated across Sections 5, 6, and 7.

    Parameters
    ----------
    panel_df : pd.DataFrame
        Long-format panel: one row per (geoid, year).
    geoid_col : str
        Column name of the geographic identifier.
    value_source_pairs : list of (str, str)
        Each tuple is (value_column, source_column).  The source column is
        updated with 'interpolated' or 'extrapolated' for filled rows.
    year_col : str
        Column name of the year integer (default 'year').

    Returns
    -------
    pd.DataFrame — same shape as input, gaps filled, source columns updated.
    """
    filled_parts = []

    for geoid, group in panel_df.groupby(geoid_col):
        group = group.sort_values(year_col).copy().reset_index(drop=True)

        for value_col, source_col in value_source_pairs:
            observed = group[[year_col, value_col]].dropna()
            original_null = group[value_col].isna().copy()

            if len(observed) >= 2:
                years_obs = observed[year_col].to_numpy(dtype=float)
                vals_obs = observed[value_col].to_numpy(dtype=float)
                first_year = int(years_obs[0])
                last_year = int(years_obs[-1])

                # 1. Fill internal gaps by linear interpolation.
                group[value_col] = group[value_col].interpolate(
                    method="linear", limit_area="inside"
                )

                # OLS trend over all observed years — used for edge extrapolation.
                slope, intercept = np.polyfit(years_obs, vals_obs, 1)

                # 2. Left-edge extrapolation.
                left_mask = group[value_col].isna() & (group[year_col] < first_year)
                if left_mask.any():
                    group.loc[left_mask, value_col] = (
                        slope * group.loc[left_mask, year_col] + intercept
                    )

                # 3. Right-edge extrapolation.
                right_mask = group[value_col].isna() & (group[year_col] > last_year)
                if right_mask.any():
                    group.loc[right_mask, value_col] = (
                        slope * group.loc[right_mask, year_col] + intercept
                    )

                # 4. Tag provenance for rows that were originally null but now filled.
                for idx in group.index:
                    if original_null.iloc[idx] and pd.notna(group.loc[idx, value_col]):
                        yr = group.loc[idx, year_col]
                        group.loc[idx, source_col] = (
                            "interpolated" if first_year < yr < last_year else "extrapolated"
                        )

            # Sign guards and rounding (applied regardless of whether fill ran).
            group[value_col] = group[value_col].round(0)
            group.loc[group[value_col] <= 0, value_col] = np.nan

        filled_parts.append(group)

    return pd.concat(filled_parts, ignore_index=True)


## 2. Load Inputs

In [6]:
# City boundary SEDF — used for export geometry and the spatial join.
# CITY_NAME is added as CITYAREA immediately so the join key is consistent throughout.
city_area_shp = pd.DataFrame.spatial.from_featureclass(
    r".\Inputs\city_area_with_workshop_areas.shp"
)

if (
    HT_INCOME_CONFIG["city_key"] not in city_area_shp.columns
    and "CITY_NAME" in city_area_shp.columns
):
    city_area_shp[HT_INCOME_CONFIG["city_key"]] = city_area_shp["CITY_NAME"]

print("City area rows:", len(city_area_shp))
city_area_shp.head()


City area rows: 109


,FID,CITY_NAME,SUBAREA,CO_NAME,SHAPE,CITYAREA
0,0,Alpine,North Utah County,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",Alpine
1,1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",Alta
2,2,American Fork,North Utah County,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",American Fork
3,3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",Balance of BOX ELDER
4,4,Benjamin,South Utah County,UTAH,"{""rings"": [[[438601.23319999967, 4437471.4559]...",Benjamin


In [7]:
# H+T cost CSV. Use thousands=',' to automatically force comma-formatted strings into numbers.
# Replace -1 sentinel with 0 (no data for that city-year).
# 'City Area' is added as CITYAREA here; 'County' is dropped.
ht_df = pd.read_csv(HT_INCOME_CONFIG["ht_csv_path"], thousands=",")
ht_df = ht_df.replace(-1, 0)

if HT_INCOME_CONFIG["city_key"] not in ht_df.columns and "City Area" in ht_df.columns:
    ht_df[HT_INCOME_CONFIG["city_key"]] = ht_df["City Area"]

if "City Area" in ht_df.columns:
    ht_df = ht_df.drop(columns=["City Area"])

# Retain County column — renamed to COUNTY_NAME for use in county income join.
if "County" in ht_df.columns:
    ht_df = ht_df.rename(columns={"County": "COUNTY_NAME"})

# Map county name → FIPS so each city row knows its parent county.
ht_df["county_geoid"] = ht_df["COUNTY_NAME"].map(HT_INCOME_CONFIG["county_name_to_fips"])

print("H+T rows:", len(ht_df))
ht_df.head()


H+T rows: 101


,COUNTY_NAME,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,CITYAREA,county_geoid
0,Utah,2255,2299,2741,3914,4274,4341,781,704,751,758,769,775,3037,3002,3492,4672,5043,5116,Alpine,49049
1,Salt Lake,0,0,0,0,0,0,603,519,555,575,588,572,0,0,0,0,0,0,Alta,49035
2,Utah,1700,1726,2045,2767,3073,3366,615,576,597,603,609,613,2315,2302,2642,3370,3682,3979,American Fork,49049
3,Box Elder,0,0,0,0,0,0,1018,857,907,970,1070,970,0,0,0,0,0,0,Balance of BOX ELDER,49003
4,Utah,0,0,0,0,0,0,694,643,675,676,679,668,0,0,0,0,0,0,Benjamin,49049


In [8]:
# Derive target years from HCOST / TCOST / HPLUST column suffixes.
target_years = sorted(
    {
        int(col[-4:])
        for col in ht_df.columns
        if col[-4:].isdigit()
        and any(col.startswith(prefix) for prefix in HT_INCOME_CONFIG["year_prefixes"])
    }
)

if HT_INCOME_CONFIG["target_years_override"] is not None:
    target_years = HT_INCOME_CONFIG["target_years_override"]

# ACS years: 2009 (earliest ACS 5-yr API vintage) through the latest H+T year.
# Used for all ACS downloads so MEDINC is available for the full income time-series.
acs_start = HT_INCOME_CONFIG["acs_start_year"]
acs_end = max(target_years)
acs_years = list(range(acs_start, acs_end + 1))

print("Detected H+T years :", target_years)
print("Using override     :", HT_INCOME_CONFIG["target_years_override"] is not None)
print(f"ACS income years   : {acs_years[0]} – {acs_years[-1]}  ({len(acs_years)} years)")


Detected H+T years : [2019, 2020, 2021, 2022, 2023, 2024]
Using override     : False
ACS income years   : 2009 – 2024  (16 years)


In [9]:
# Validation: confirm city names in CSV match city names in shapefile.
ht_cities = set(ht_df[HT_INCOME_CONFIG["city_key"]].dropna())
shp_col = (
    HT_INCOME_CONFIG["city_key"]
    if HT_INCOME_CONFIG["city_key"] in city_area_shp.columns
    else "CITY_NAME"
)
shape_cities = set(city_area_shp[shp_col].dropna())

in_csv_not_shp = sorted(ht_cities - shape_cities)
in_shp_not_csv = sorted(shape_cities - ht_cities)

print("In CSV but NOT shapefile (investigate if non-empty):", in_csv_not_shp)
print("In shapefile but NOT CSV (geometry-only rows, expected):", in_shp_not_csv)


In CSV but NOT shapefile (investigate if non-empty): []
In shapefile but NOT CSV (geometry-only rows, expected): ['Camp Williams', 'Davis County', 'Lake Mountain', 'SL County East Cyns', 'South Cedar Valley', 'South Goshen Valley', 'Utah Lake', 'West Mountain']


## 3. Build 2020 Block Group Reference

A single 2020 Census block group layer is used as the stable spatial reference for all
target years.  It is fetched once via `pygris`, normalised to `MultiPolygon` (required by
the OpenFileGDB Fiona driver), and persisted to a file GDB.  Subsequent runs skip the
fetch if the layer already exists.

In [10]:
bg_reference_fc = os.path.join(
    HT_INCOME_CONFIG["bg_reference_gdb"], HT_INCOME_CONFIG["bg_reference_name"]
)

if not arcpy.Exists(HT_INCOME_CONFIG["bg_reference_gdb"]):
    arcpy.CreateFileGDB_management(r".\Inputs", "block_groups_2020.gdb")

if not arcpy.Exists(bg_reference_fc):
    bg_2020_gdf = block_groups(
        state=HT_INCOME_CONFIG["state"], year=2020, cache=HT_INCOME_CONFIG["pygris_cache"]
    )
    bg_2020_gdf = bg_2020_gdf[bg_2020_gdf["COUNTYFP"].isin(HT_INCOME_CONFIG["county_fips"])].copy()
    bg_2020_gdf = bg_2020_gdf[
        ["GEOID", "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE", "geometry"]
    ].copy()

    # Normalise mixed Polygon/MultiPolygon so the OpenFileGDB driver has a consistent type.
    bg_2020_gdf["geometry"] = bg_2020_gdf["geometry"].apply(
        lambda g: MultiPolygon([g]) if isinstance(g, Polygon) else g
    )
    bg_2020_gdf.to_file(
        HT_INCOME_CONFIG["bg_reference_gdb"],
        layer=HT_INCOME_CONFIG["bg_reference_name"],
        driver="OpenFileGDB",
    )
    print("Exported:", bg_reference_fc)
    print("Rows    :", len(bg_2020_gdf))
    print("Counties:", sorted(bg_2020_gdf["COUNTYFP"].unique().tolist()))
    print("CRS     :", bg_2020_gdf.crs)
else:
    print("Using existing block group reference:", bg_reference_fc)


Using existing block group reference: .\Inputs\block_groups_2020.gdb\bg_2020_ut_5county


In [11]:
# Read back as arcgis SEDF.  Cast GEOID to str — all downstream joins depend on this.
block_groups_ref = pd.DataFrame.spatial.from_featureclass(bg_reference_fc)
block_groups_ref[HT_INCOME_CONFIG["bg_key"]] = block_groups_ref[HT_INCOME_CONFIG["bg_key"]].astype(
    str
)

print("Columns:", block_groups_ref.columns.tolist())
print("Shape  :", block_groups_ref.shape)
block_groups_ref.head()


Columns: ['OBJECTID', 'GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'SHAPE']
Shape  : (1547, 7)


,OBJECTID,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,SHAPE
0,1,490351113061,49,035,111306,1,"{""rings"": [[[-111.83382099999994, 40.615462000..."
1,2,490351113062,49,035,111306,2,"{""rings"": [[[-111.82158699999997, 40.607759000..."
2,3,490351114001,49,035,111400,1,"{""rings"": [[[-111.88258799999994, 40.718417000..."
3,4,490351114002,49,035,111400,2,"{""rings"": [[[-111.88262299999997, 40.712825000..."
4,5,490351114003,49,035,111400,3,"{""rings"": [[[-111.88267499999995, 40.704964000..."


## 4. Fetch ACS Block Group Data by Year

For each target year, `get_census` retrieves median household income (`B19013_001E`),
total population (`B02001_001E`), and total households (`B11001_001E`) at the block-group
level across all 5 counties.  ACS data is left-joined onto the 2020 BG reference geometry.

Years where the API call fails are logged but do not stop execution — missing years are
handled in Section 5.

In [12]:
bg_acs_by_year = {}
acs_year_status = []

for year in acs_years:
    county_frames = []
    year_ok = True
    year_message = "ok"

    for county_fips in HT_INCOME_CONFIG["county_fips"]:
        try:
            county_df = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                year=year,
                params={
                    "for": "block group:*",
                    "in": (f"state:{HT_INCOME_CONFIG['state_fips']} county:{county_fips}"),
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            county_frames.append(county_df)
        except Exception as exc:
            year_ok = False
            year_message = str(exc)
            break

    if year_ok and county_frames:
        acs_df = pd.concat(county_frames, ignore_index=True)
        acs_df = acs_df.rename(
            columns={
                HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
            }
        )
        acs_df[HT_INCOME_CONFIG["bg_key"]] = acs_df[HT_INCOME_CONFIG["bg_key"]].astype(str)

        for col in ("median_income", "population", "households"):
            acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")
            acs_df.loc[acs_df[col] <= 0, col] = np.nan

        acs_df["acs_year"] = year
        acs_df["income_source"] = "acs"
        acs_df["population_source"] = "acs"
        acs_df["households_source"] = "acs"

        acs_df = acs_df[
            [
                HT_INCOME_CONFIG["bg_key"],
                "median_income",
                "population",
                "households",
                "acs_year",
                "income_source",
                "population_source",
                "households_source",
            ]
        ].copy()

        bg_acs_by_year[year] = block_groups_ref.merge(
            acs_df, on=HT_INCOME_CONFIG["bg_key"], how="left"
        )
        acs_year_status.append(
            {
                "year": year,
                "status": "fetched",
                "acs_rows": len(acs_df),
                "matched_inc_rows": bg_acs_by_year[year]["median_income"].notna().sum(),
                "matched_hh_rows": bg_acs_by_year[year]["households"].notna().sum(),
                "message": "ok",
            }
        )
    else:
        bg_acs_by_year[year] = block_groups_ref.copy()
        for col in ("median_income", "population", "households"):
            bg_acs_by_year[year][col] = np.nan
        bg_acs_by_year[year]["acs_year"] = year
        bg_acs_by_year[year]["income_source"] = np.nan
        bg_acs_by_year[year]["population_source"] = np.nan
        bg_acs_by_year[year]["households_source"] = np.nan
        acs_year_status.append(
            {
                "year": year,
                "status": "missing",
                "acs_rows": 0,
                "matched_inc_rows": 0,
                "matched_hh_rows": 0,
                "message": year_message,
            }
        )


In [13]:
# Fetch summary — check for any missing years before proceeding.
acs_year_status_df = pd.DataFrame(acs_year_status)
acs_year_status_df


,year,status,acs_rows,matched_inc_rows,matched_hh_rows,message
0,2009,missing,0,0,0,Request failed. The Census Bureau error messag...
1,2010,missing,0,0,0,Request failed. The Census Bureau error messag...
2,2011,missing,0,0,0,Request failed. The Census Bureau error messag...
3,2012,missing,0,0,0,Request failed. The Census Bureau error messag...
4,2013,fetched,1297,1058,1058,ok
5,2014,fetched,1297,1057,1058,ok
6,2015,fetched,1297,1051,1057,ok
7,2016,fetched,1297,1049,1057,ok
8,2017,fetched,1297,1052,1057,ok
9,2018,fetched,1297,1054,1057,ok


In [14]:
# Spot check: confirm ACS fields joined correctly for the first year.
sample_year = target_years[0]
print("Sample year:", sample_year)
print("Shape      :", bg_acs_by_year[sample_year].shape)
bg_acs_by_year[sample_year][
    [
        HT_INCOME_CONFIG["bg_key"],
        "median_income",
        "population",
        "households",
        "acs_year",
        "income_source",
    ]
].head()


Sample year: 2019
Shape      : (1547, 14)


,GEOID,median_income,population,households,acs_year,income_source
0,490351113061,75438.0,1792.0,723.0,2019.0,acs
1,490351113062,128194.0,839.0,335.0,2019.0,acs
2,490351114001,68387.0,1330.0,409.0,2019.0,acs
3,490351114002,78548.0,1086.0,396.0,2019.0,acs
4,490351114003,51789.0,1734.0,730.0,2019.0,acs


## 5. Estimate Missing BG ACS Years

Each block group is processed independently across all target years using the
`fill_acs_panel` helper defined in Section 1:

- **Internal gaps** (e.g. 2020 missing when 2019 and 2021 exist) are filled by linear
  interpolation between the two nearest flanking observed values.
- **Edge gaps** (leading or trailing years) are filled by linear extrapolation using an
  OLS trend fitted to **all** observed years for that block group.
- Block groups with fewer than 2 observed years are left as `NaN`.

Applies to `median_income`, `population`, and `households`.  Observed ACS values are
never overwritten.  Provenance is tracked via `income_source`, `population_source`, and
`households_source`: `"acs"` | `"interpolated"` | `"extrapolated"`.

In [15]:
# Build a long panel (one row per GEOID x year) to drive the fill logic.
acs_panel_parts = []
for year in acs_years:
    year_df = bg_acs_by_year[year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ].copy()
    year_df["year"] = year
    acs_panel_parts.append(year_df)

acs_panel_df = pd.concat(acs_panel_parts, ignore_index=True)
print(f"Panel shape: {acs_panel_df.shape}  ({len(acs_years)} years x {len(block_groups_ref)} BGs)")
acs_panel_df.head()


Panel shape: (24752, 8)  (16 years x 1547 BGs)


,GEOID,median_income,population,households,income_source,population_source,households_source,year
0,490351113061,NaN,NaN,NaN,NaN,NaN,NaN,2009
1,490351113062,NaN,NaN,NaN,NaN,NaN,NaN,2009
2,490351114001,NaN,NaN,NaN,NaN,NaN,NaN,2009
3,490351114002,NaN,NaN,NaN,NaN,NaN,NaN,2009
4,490351114003,NaN,NaN,NaN,NaN,NaN,NaN,2009


In [16]:
# Run fill_acs_panel on the block-group panel.
# The helper handles all three variables in one pass — see Section 1 for the full docstring.
acs_panel_filled_df = fill_acs_panel(
    acs_panel_df,
    geoid_col=HT_INCOME_CONFIG["bg_key"],
    value_source_pairs=[
        ("median_income", "income_source"),
        ("population", "population_source"),
        ("households", "households_source"),
    ],
)
print("Filled panel shape:", acs_panel_filled_df.shape)
acs_panel_filled_df.head()


Filled panel shape: (24752, 8)


,GEOID,median_income,population,households,income_source,population_source,households_source,year
0,490039601001,42861.0,507.0,187.0,extrapolated,extrapolated,extrapolated,2009
1,490039601001,46114.0,554.0,197.0,extrapolated,extrapolated,extrapolated,2010
2,490039601001,49367.0,601.0,207.0,extrapolated,extrapolated,extrapolated,2011
3,490039601001,52620.0,648.0,217.0,extrapolated,extrapolated,extrapolated,2012
4,490039601001,65000.0,698.0,228.0,acs,acs,acs,2013


In [17]:
# Fill provenance summary by year.
fill_summary_df = (
    acs_panel_filled_df.groupby("year")
    .agg(
        income_acs=pd.NamedAgg(column="income_source", aggfunc=lambda s: (s == "acs").sum()),
        income_interpolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        income_extrapolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
        pop_acs=pd.NamedAgg(column="population_source", aggfunc=lambda s: (s == "acs").sum()),
        pop_interpolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        pop_extrapolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
        hh_acs=pd.NamedAgg(column="households_source", aggfunc=lambda s: (s == "acs").sum()),
        hh_interpolated=pd.NamedAgg(
            column="households_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        hh_extrapolated=pd.NamedAgg(
            column="households_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
    )
    .reset_index()
)
fill_summary_df


,year,income_acs,income_interpolated,income_extrapolated,pop_acs,pop_interpolated,pop_extrapolated,hh_acs,hh_interpolated,hh_extrapolated
0,2009,0,0,1528,0,0,1535,0,0,1533
1,2010,0,0,1528,0,0,1535,0,0,1533
2,2011,0,0,1528,0,0,1535,0,0,1533
3,2012,0,0,1528,0,0,1535,0,0,1533
4,2013,1062,0,470,1063,0,474,1061,0,475
5,2014,1061,1,470,1063,0,474,1061,0,475
6,2015,1055,7,470,1062,1,474,1060,1,475
7,2016,1053,9,470,1063,0,474,1060,1,475
8,2017,1056,6,470,1063,0,474,1060,1,475
9,2018,1058,4,470,1063,0,474,1060,1,475


In [18]:
# Spot check: all years for one GEOID should have sensible values and sources.
sample_geoid = acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]].iloc[0]
acs_panel_filled_df[acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]] == sample_geoid].sort_values(
    "year"
)[["year", "median_income", "population", "households", "income_source", "households_source"]]


,year,median_income,population,households,income_source,households_source
0,2009,42861.0,507.0,187.0,extrapolated,extrapolated
1,2010,46114.0,554.0,197.0,extrapolated,extrapolated
2,2011,49367.0,601.0,207.0,extrapolated,extrapolated
3,2012,52620.0,648.0,217.0,extrapolated,extrapolated
4,2013,65000.0,698.0,228.0,acs,acs
5,2014,67361.0,769.0,248.0,acs,acs
6,2015,62237.0,803.0,250.0,acs,acs
7,2016,61544.0,885.0,259.0,acs,acs
8,2017,63047.0,887.0,277.0,acs,acs
9,2018,66000.0,900.0,282.0,acs,acs


In [19]:
# Write filled values back into bg_acs_by_year, replacing the raw ACS columns.
for year in acs_years:
    year_fill = acs_panel_filled_df[acs_panel_filled_df["year"] == year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ].copy()

    base_cols = [
        col
        for col in bg_acs_by_year[year].columns
        if col
        not in [
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ]
    bg_acs_by_year[year] = bg_acs_by_year[year][base_cols].merge(
        year_fill, on=HT_INCOME_CONFIG["bg_key"], how="left"
    )


## 6. Fetch & Cache Tract-Level ACS Fallback

Census tract data is virtually never suppressed for `B19013_001E` because tracts are
designed to contain 2,500–8,000 people — large enough for reliable ACS estimates.  This
section downloads tract-level income, population, and household counts for all 5 counties,
applies the same `fill_acs_panel` interpolation/extrapolation pipeline used for block
groups, and saves the filled panel to a CSV.

**File:** `.\Inputs\tracts_2020_acs_income.csv`  
**Skipped:** if the file already exists.  Delete to force a full re-download.

In [20]:
tract_ref_path = HT_INCOME_CONFIG["tract_ref_path"]

if os.path.exists(tract_ref_path):
    tract_acs_filled_df = pd.read_csv(tract_ref_path, dtype={HT_INCOME_CONFIG["tract_key"]: str})
    print("Loaded cached tract ACS data:", tract_ref_path)
    print("Shape:", tract_acs_filled_df.shape)
else:
    print("Fetching tract-level ACS data for all target years...")
    tract_raw_parts = []
    tract_year_status = []

    for year in acs_years:
        county_frames = []
        year_ok = True
        year_message = "ok"

        for county_fips in HT_INCOME_CONFIG["county_fips"]:
            try:
                county_df = get_census(
                    dataset=HT_INCOME_CONFIG["acs_dataset"],
                    variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                    year=year,
                    params={
                        "for": "tract:*",
                        "in": (f"state:{HT_INCOME_CONFIG['state_fips']} county:{county_fips}"),
                        "key": CENSUS_API_KEY,
                    },
                    return_geoid=True,
                    guess_dtypes=True,
                )
                county_frames.append(county_df)
            except Exception as exc:
                year_ok = False
                year_message = str(exc)
                break

        if year_ok and county_frames:
            acs_df = pd.concat(county_frames, ignore_index=True)
            acs_df = acs_df.rename(
                columns={
                    HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                    HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                    HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
                }
            )
            acs_df[HT_INCOME_CONFIG["tract_key"]] = acs_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
            for col in ("median_income", "population", "households"):
                acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")
                acs_df.loc[acs_df[col] <= 0, col] = np.nan

            acs_df["year"] = year
            acs_df["income_source"] = "acs"
            acs_df["population_source"] = "acs"
            acs_df["households_source"] = "acs"

            tract_raw_parts.append(
                acs_df[
                    [
                        HT_INCOME_CONFIG["tract_key"],
                        "median_income",
                        "population",
                        "households",
                        "year",
                        "income_source",
                        "population_source",
                        "households_source",
                    ]
                ]
            )
            tract_year_status.append(
                {"year": year, "status": "fetched", "rows": len(acs_df), "message": "ok"}
            )
        else:
            tract_year_status.append(
                {"year": year, "status": "missing", "rows": 0, "message": year_message}
            )

    print(pd.DataFrame(tract_year_status))

    if not tract_raw_parts:
        raise RuntimeError(
            "No tract ACS data was fetched. Check CENSUS_API_KEY and target year range."
        )

    tract_panel_df = pd.concat(tract_raw_parts, ignore_index=True)
    print(f"Raw tract panel: {tract_panel_df.shape}")

    tract_acs_filled_df = fill_acs_panel(
        tract_panel_df,
        geoid_col=HT_INCOME_CONFIG["tract_key"],
        value_source_pairs=[
            ("median_income", "income_source"),
            ("population", "population_source"),
            ("households", "households_source"),
        ],
    )

    tract_acs_filled_df.to_csv(tract_ref_path, index=False)
    print("Saved:", tract_ref_path)
    print("Shape:", tract_acs_filled_df.shape)


Fetching tract-level ACS data for all target years...
    year   status  rows message
0   2009  fetched   379      ok
1   2010  fetched   455      ok
2   2011  fetched   455      ok
3   2012  fetched   455      ok
4   2013  fetched   455      ok
5   2014  fetched   455      ok
6   2015  fetched   455      ok
7   2016  fetched   455      ok
8   2017  fetched   455      ok
9   2018  fetched   455      ok
10  2019  fetched   455      ok
11  2020  fetched   543      ok
12  2021  fetched   543      ok
13  2022  fetched   543      ok
14  2023  fetched   543      ok
15  2024  fetched   543      ok
Raw tract panel: (7644, 8)
Saved: .\Inputs\tracts_2020_acs_income.csv
Shape: (7644, 8)


In [21]:
# Tract fill provenance spot check.
tract_fill_summary = (
    tract_acs_filled_df.groupby("year")
    .agg(
        total=("median_income", "count"),
        null_income=("median_income", lambda s: s.isna().sum()),
        income_acs=("income_source", lambda s: (s == "acs").sum()),
        income_interp=("income_source", lambda s: (s == "interpolated").sum()),
        income_extrap=("income_source", lambda s: (s == "extrapolated").sum()),
    )
    .reset_index()
)
print("Tract ACS fill summary:")
tract_fill_summary


Tract ACS fill summary:


,year,total,null_income,income_acs,income_interp,income_extrap
0,2009,375,4,379,0,0
1,2010,452,3,455,0,0
2,2011,452,3,455,0,0
3,2012,452,3,455,0,0
4,2013,452,3,455,0,0
5,2014,452,3,455,0,0
6,2015,452,3,455,0,0
7,2016,452,3,455,0,0
8,2017,452,3,455,0,0
9,2018,452,3,455,0,0


## 7. Fetch & Cache County-Level ACS Fallback

County-level ACS income is always available and robust (very large sample).  This section
provides the last-resort fallback for block groups whose income could not be filled from
the parent tract.  The same `fill_acs_panel` pipeline is applied and results are cached.

**File:** `.\Inputs\counties_2020_acs_income.csv`  
**Skipped:** if the file already exists.  Delete to force a full re-download.

In [22]:
county_ref_path = HT_INCOME_CONFIG["county_ref_path"]

if os.path.exists(county_ref_path):
    county_acs_filled_df = pd.read_csv(county_ref_path, dtype={HT_INCOME_CONFIG["county_key"]: str})
    print("Loaded cached county ACS data:", county_ref_path)
    print("Shape:", county_acs_filled_df.shape)
else:
    print("Fetching county-level ACS data for all target years...")
    county_raw_parts = []
    county_year_status = []

    for year in acs_years:
        try:
            raw_df = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                year=year,
                params={
                    "for": "county:*",
                    "in": f"state:{HT_INCOME_CONFIG['state_fips']}",
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            # Derive county GEOID and filter to our 5 counties.
            raw_df[HT_INCOME_CONFIG["county_key"]] = (
                raw_df[HT_INCOME_CONFIG["bg_key"]].astype(str).str[:5]
            )
            raw_df = raw_df[
                raw_df[HT_INCOME_CONFIG["county_key"]].str[2:].isin(HT_INCOME_CONFIG["county_fips"])
            ].copy()

            raw_df = raw_df.rename(
                columns={
                    HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                    HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                    HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
                }
            )
            for col in ("median_income", "population", "households"):
                raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")
                raw_df.loc[raw_df[col] <= 0, col] = np.nan

            raw_df["year"] = year
            raw_df["income_source"] = "acs"
            raw_df["population_source"] = "acs"
            raw_df["households_source"] = "acs"

            county_raw_parts.append(
                raw_df[
                    [
                        HT_INCOME_CONFIG["county_key"],
                        "median_income",
                        "population",
                        "households",
                        "year",
                        "income_source",
                        "population_source",
                        "households_source",
                    ]
                ]
            )
            county_year_status.append(
                {"year": year, "status": "fetched", "rows": len(raw_df), "message": "ok"}
            )
        except Exception as exc:
            county_year_status.append(
                {"year": year, "status": "missing", "rows": 0, "message": str(exc)}
            )

    print(pd.DataFrame(county_year_status))

    if not county_raw_parts:
        raise RuntimeError(
            "No county ACS data was fetched. Check CENSUS_API_KEY and target year range."
        )

    county_panel_df = pd.concat(county_raw_parts, ignore_index=True)
    print(f"Raw county panel: {county_panel_df.shape}")

    county_acs_filled_df = fill_acs_panel(
        county_panel_df,
        geoid_col=HT_INCOME_CONFIG["county_key"],
        value_source_pairs=[
            ("median_income", "income_source"),
            ("population", "population_source"),
            ("households", "households_source"),
        ],
    )

    county_acs_filled_df.to_csv(county_ref_path, index=False)
    print("Saved:", county_ref_path)
    print("Shape:", county_acs_filled_df.shape)


Fetching county-level ACS data for all target years...
    year   status  rows message
0   2009  fetched     5      ok
1   2010  fetched     5      ok
2   2011  fetched     5      ok
3   2012  fetched     5      ok
4   2013  fetched     5      ok
5   2014  fetched     5      ok
6   2015  fetched     5      ok
7   2016  fetched     5      ok
8   2017  fetched     5      ok
9   2018  fetched     5      ok
10  2019  fetched     5      ok
11  2020  fetched     5      ok
12  2021  fetched     5      ok
13  2022  fetched     5      ok
14  2023  fetched     5      ok
15  2024  fetched     5      ok
Raw county panel: (80, 8)
Saved: .\Inputs\counties_2020_acs_income.csv
Shape: (80, 8)


In [23]:
# County-level data should have no null income after fill.
print("County ACS filled data:")
county_acs_filled_df[
    [HT_INCOME_CONFIG["county_key"], "year", "median_income", "income_source"]
].sort_values([HT_INCOME_CONFIG["county_key"], "year"])


County ACS filled data:


,county_geoid,year,median_income,income_source
0,49003,2009,54670.0,acs
1,49003,2010,55135.0,acs
2,49003,2011,55588.0,acs
3,49003,2012,55918.0,acs
4,49003,2013,57292.0,acs
...,...,...,...,...
75,49057,2020,71275.0,acs
76,49057,2021,74345.0,acs
77,49057,2022,82291.0,acs
78,49057,2023,87083.0,acs


## 8. Apply Hierarchical Income Fallback (BG → Tract → County)

For every block group that still has a null `median_income` after Section 5 (whether due
to ACS suppression, API failure, or no observed years), this section fills the value from:

1. **Tract level** — the population-weighted mean income of the parent census tract
   (derived as the first 11 characters of the block group GEOID).
2. **County level** — the population-weighted mean income of the parent county
   (derived as the first 5 characters of the block group GEOID), used only when the
   tract value is also unavailable.

Only `median_income` and `income_source` are updated; `population` and `households`
remain as fetched by the ACS.  Cities with no block groups assigned (non-residential
TDM zones such as Utah Lake or Camp Williams) will still have null city-level income
after aggregation — this is correct behaviour and those cities receive `MEDINC = 0`
in the final export.

In [24]:
fallback_summary = []

for year in acs_years:
    # ── Lookup tables: one income value per tract / county for this year ──────
    tract_lookup = (
        tract_acs_filled_df[tract_acs_filled_df["year"] == year][
            [HT_INCOME_CONFIG["tract_key"], "median_income"]
        ]
        .rename(columns={"median_income": "tract_income"})
        .drop_duplicates(HT_INCOME_CONFIG["tract_key"])
    )
    county_lookup = (
        county_acs_filled_df[county_acs_filled_df["year"] == year][
            [HT_INCOME_CONFIG["county_key"], "median_income"]
        ]
        .rename(columns={"median_income": "county_income"})
        .drop_duplicates(HT_INCOME_CONFIG["county_key"])
    )

    # ── Attach parent-geography keys to each BG row ───────────────────────────
    bg_df = bg_acs_by_year[year].copy()
    bg_df[HT_INCOME_CONFIG["bg_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
    bg_df[HT_INCOME_CONFIG["tract_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].str[:11]
    bg_df[HT_INCOME_CONFIG["county_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].str[:5]

    bg_df = bg_df.merge(tract_lookup, on=HT_INCOME_CONFIG["tract_key"], how="left")
    bg_df = bg_df.merge(county_lookup, on=HT_INCOME_CONFIG["county_key"], how="left")

    n_before = int(bg_df["median_income"].isna().sum())

    # ── Tract fallback ─────────────────────────────────────────────────────────
    null_mask = bg_df["median_income"].isna()
    tract_avail = null_mask & bg_df["tract_income"].notna()
    bg_df.loc[tract_avail, "median_income"] = bg_df.loc[tract_avail, "tract_income"]
    bg_df.loc[tract_avail, "income_source"] = "tract_fallback"
    n_tract = int(tract_avail.sum())

    # ── County fallback ────────────────────────────────────────────────────────
    null_mask = bg_df["median_income"].isna()
    county_avail = null_mask & bg_df["county_income"].notna()
    bg_df.loc[county_avail, "median_income"] = bg_df.loc[county_avail, "county_income"]
    bg_df.loc[county_avail, "income_source"] = "county_fallback"
    n_county = int(county_avail.sum())

    n_after = int(bg_df["median_income"].isna().sum())
    fallback_summary.append(
        {
            "year": year,
            "null_before": n_before,
            "filled_by_tract": n_tract,
            "filled_by_county": n_county,
            "still_null": n_after,
        }
    )

    # ── Write updated income / source back; drop temporary join columns ────────
    update_df = bg_df[[HT_INCOME_CONFIG["bg_key"], "median_income", "income_source"]].copy()
    base_cols = [
        c for c in bg_acs_by_year[year].columns if c not in ("median_income", "income_source")
    ]
    bg_acs_by_year[year] = bg_acs_by_year[year][base_cols].merge(
        update_df, on=HT_INCOME_CONFIG["bg_key"], how="left"
    )

print("Hierarchical fallback complete.")
pd.DataFrame(fallback_summary)


Hierarchical fallback complete.


,year,null_before,filled_by_tract,filled_by_county,still_null
0,2009,224,38,186,0
1,2010,196,46,150,0
2,2011,167,35,132,0
3,2012,132,27,105,0
4,2013,108,21,87,0
5,2014,85,17,68,0
6,2015,69,15,54,0
7,2016,45,11,34,0
8,2017,28,5,23,0
9,2018,23,4,19,0


In [25]:
# Provenance breakdown after fallback: how many BGs got each income source?
source_summary = []
for year in acs_years:
    counts = bg_acs_by_year[year]["income_source"].value_counts(dropna=False)
    row = {"year": year}
    for src in ("acs", "interpolated", "extrapolated", "tract_fallback", "county_fallback"):
        row[src] = int(counts.get(src, 0))
    row["still_null"] = int(bg_acs_by_year[year]["median_income"].isna().sum())
    source_summary.append(row)
pd.DataFrame(source_summary)


,year,acs,interpolated,extrapolated,tract_fallback,county_fallback,still_null
0,2009,0,0,1323,38,186,0
1,2010,0,0,1351,46,150,0
2,2011,0,0,1380,35,132,0
3,2012,0,0,1415,27,105,0
4,2013,1058,0,381,21,87,0
5,2014,1057,1,404,17,68,0
6,2015,1051,7,420,15,54,0
7,2016,1049,9,444,11,34,0
8,2017,1052,6,461,5,23,0
9,2018,1054,4,466,4,19,0


## 9. Assign Block Groups to Cities (Spatial Join)

Block groups are assigned to exactly one city using a centroid-based spatial join
(`HAVE_THEIR_CENTER_IN`).  This avoids double-counting from polygon-intersection joins.
Unmatched block groups are logged and excluded from aggregation.

The BG reference is in GCS NAD83 (EPSG:4269); the city shapefile is in UTM Zone 12N
(EPSG:26912).  A `CopyFeatures` + `Project` pattern re-projects the BGs — `CopyFeatures`
is required first to avoid ERROR 001489 (topology participation prevents direct `Project`).

In [26]:
city_fc = r".\Inputs\city_area_with_workshop_areas.shp"
bg_copy_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"] + "_copy")
bg_projected_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"])
bg_city_join_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_city_join_name"])
bg_join_input_fc = bg_reference_fc

bg_sr = arcpy.Describe(bg_reference_fc).spatialReference
city_sr = arcpy.Describe(city_fc).spatialReference

print("BG SR  :", bg_sr.name, bg_sr.factoryCode)
print("City SR:", city_sr.name, city_sr.factoryCode)

if bg_sr.factoryCode != city_sr.factoryCode:
    if arcpy.Exists(bg_copy_fc):
        arcpy.management.Delete(bg_copy_fc)
    arcpy.management.CopyFeatures(bg_reference_fc, bg_copy_fc)

    if arcpy.Exists(bg_projected_fc):
        arcpy.management.Delete(bg_projected_fc)
    arcpy.management.Project(
        in_dataset=bg_copy_fc, out_dataset=bg_projected_fc, out_coor_system=city_sr
    )
    arcpy.management.Delete(bg_copy_fc)

    bg_join_input_fc = bg_projected_fc
    print("Projected to:", city_sr.name)
else:
    print("CRS match — no projection needed")


BG SR  : GCS_North_American_1983 4269
City SR: NAD_1983_UTM_Zone_12N 26912
Projected to: NAD_1983_UTM_Zone_12N


In [27]:
if arcpy.Exists(bg_city_join_fc):
    arcpy.management.Delete(bg_city_join_fc)

arcpy.analysis.SpatialJoin(
    target_features=bg_join_input_fc,
    join_features=city_fc,
    out_feature_class=bg_city_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="HAVE_THEIR_CENTER_IN",
)


<Result '.\\Outputs\\scratch.gdb\\bg_city_spatial_join'>

In [28]:
# Read back the join result; apply workshop-area renames to SUBAREA.
bg_city_lookup = pd.DataFrame.spatial.from_featureclass(bg_city_join_fc)[
    ["GEOID", "CITY_NAME", "SUBAREA", "CO_NAME"]
].copy()
bg_city_lookup.rename(columns={"CITY_NAME": HT_INCOME_CONFIG["city_key"]}, inplace=True)
bg_city_lookup[HT_INCOME_CONFIG["bg_key"]] = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].astype(str)
bg_city_lookup["CO_NAME"] = bg_city_lookup["CO_NAME"].str.upper()
bg_city_lookup["SUBAREA"] = bg_city_lookup["SUBAREA"].replace(
    HT_INCOME_CONFIG["workshop_area_lookup"]
)

bg_city_lookup.head()


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE


In [29]:
# Validate: duplicate GEOIDs indicate a join error and must be zero.
dup_count = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].duplicated().sum()
if dup_count > 0:
    raise ValueError(
        f"Duplicate GEOIDs in bg_city_lookup: {dup_count}. Investigate before continuing."
    )

print("Total BG rows    :", len(bg_city_lookup))
print("Duplicate GEOIDs :", dup_count)
print("Matched BGs      :", bg_city_lookup[HT_INCOME_CONFIG["city_key"]].notna().sum())
print(
    "Unmatched BGs    :",
    bg_city_lookup[HT_INCOME_CONFIG["city_key"]].isna().sum(),
    "(excluded from aggregation — expected for non-residential TDM zones)",
)

bg_city_lookup[
    [HT_INCOME_CONFIG["bg_key"], HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME"]
].head(10)


Total BG rows    : 1547
Duplicate GEOIDs : 0
Matched BGs      : 1521
Unmatched BGs    : 26 (excluded from aggregation — expected for non-residential TDM zones)


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE
5,490351114005,South Salt Lake,Salt Lake County North,SALT LAKE
6,490351114006,South Salt Lake,Salt Lake County North,SALT LAKE
7,490572002022,Ogden,Weber County South,WEBER
8,490351117023,South Salt Lake,Salt Lake County North,SALT LAKE
9,490351117024,South Salt Lake,Salt Lake County North,SALT LAKE


## 10. Aggregate to City-Level Income

For each year, block-group income and household counts are aggregated to the city level:

```
MEDINC_YYYY = sum(median_income × households) / sum(households)
TOTHH_YYYY  = sum(households)                              [for all matched BGs]
TOTPOP_YYYY = sum(population)                              [for all matched BGs]
```

Block groups with missing income, missing households, or zero households are excluded
from the weighted income calculation.  `TOTHH` and `TOTPOP` sums include only those
same contributing block groups for internal consistency.

In [30]:
city_income_parts = []

for year in acs_years:
    year_df = bg_acs_by_year[year].copy()
    year_df[HT_INCOME_CONFIG["bg_key"]] = year_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
    year_df = year_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")

    # Keep only rows that can contribute to the weighted average.
    year_df = year_df[
        year_df[HT_INCOME_CONFIG["city_key"]].notna()
        & year_df["median_income"].notna()
        & year_df["households"].notna()
        & (year_df["households"] > 0)
    ].copy()

    year_df["weighted_income"] = year_df["median_income"] * year_df["households"]

    city_year = (
        year_df.groupby(HT_INCOME_CONFIG["city_key"], dropna=False)
        .agg(
            **{
                f"TOTPOP{year}": ("population", "sum"),
                f"TOTHH{year}": ("households", "sum"),
                f"MEDINC{year}": ("weighted_income", "sum"),
                f"bg_count_{year}": (HT_INCOME_CONFIG["bg_key"], "count"),
            }
        )
        .reset_index()
    )
    # Convert weighted sum → weighted mean.
    city_year[f"MEDINC{year}"] = (city_year[f"MEDINC{year}"] / city_year[f"TOTHH{year}"]).round(0)

    city_income_parts.append(city_year)


In [31]:
# Merge all year slices into one wide DataFrame keyed on CITYAREA.
# city_income_parts was built over acs_years; merge all slices.
city_income_df = city_income_parts[0].copy()
for part in city_income_parts[1:]:
    city_income_df = city_income_df.merge(part, on=HT_INCOME_CONFIG["city_key"], how="outer")

print("Cities with any income data:", len(city_income_df))
print("Duplicate city rows:", city_income_df[HT_INCOME_CONFIG["city_key"]].duplicated().sum())
city_income_df.head(10)


Cities with any income data: 90
Duplicate city rows: 0


,CITYAREA,TOTPOP2009,TOTHH2009,MEDINC2009,bg_count_2009,TOTPOP2010,TOTHH2010,MEDINC2010,bg_count_2010,TOTPOP2011,TOTHH2011,MEDINC2011,bg_count_2011,TOTPOP2012,TOTHH2012,MEDINC2012,bg_count_2012,TOTPOP2013,TOTHH2013,MEDINC2013,bg_count_2013,TOTPOP2014,TOTHH2014,MEDINC2014,bg_count_2014,TOTPOP2015,TOTHH2015,MEDINC2015,bg_count_2015,TOTPOP2016,TOTHH2016,MEDINC2016,bg_count_2016,TOTPOP2017,TOTHH2017,MEDINC2017,bg_count_2017,TOTPOP2018,TOTHH2018,MEDINC2018,bg_count_2018,TOTPOP2019,TOTHH2019,MEDINC2019,bg_count_2019,TOTPOP2020,TOTHH2020,MEDINC2020,bg_count_2020,TOTPOP2021,TOTHH2021,MEDINC2021,bg_count_2021,TOTPOP2022,TOTHH2022,MEDINC2022,bg_count_2022,TOTPOP2023,TOTHH2023,MEDINC2023,bg_count_2023,TOTPOP2024,TOTHH2024,MEDINC2024,bg_count_2024
0,Alpine,9313.0,2263.0,80435.0,7.0,9349.0,2291.0,86335.0,7.0,9381.0,2320.0,92255.0,7.0,9415.0,2352.0,98036.0,7.0,9409.0,2365.0,115225.0,7.0,9159.0,2396.0,107480.0,7.0,9250.0,2516.0,102811.0,7.0,9686.0,2468.0,115093.0,7.0,9692.0,2472.0,127079.0,7.0,9924.0,2557.0,137687.0,7.0,9794.0,2567.0,132896.0,7.0,10208.0,2627.0,139065.0,7,9756.0,2552.0,143451.0,7,9484.0,2554.0,164232.0,7,9649.0,2697.0,156122.0,7,9566.0,2796.0,159992.0,7
1,American Fork,27246.0,7171.0,48437.0,18.0,27456.0,7307.0,51177.0,19.0,27537.0,7451.0,47695.0,19.0,27614.0,7595.0,52018.0,19.0,26935.0,7707.0,60587.0,19.0,27518.0,7856.0,62826.0,19.0,28153.0,8101.0,62190.0,19.0,28621.0,8049.0,64775.0,20.0,29599.0,8426.0,68317.0,21.0,31965.0,8970.0,72977.0,22.0,33624.0,9733.0,79718.0,22.0,33780.0,9799.0,82372.0,22,34725.0,10334.0,86672.0,22,36227.0,10841.0,96642.0,22,37988.0,11642.0,103455.0,22,39785.0,12326.0,106438.0,22
2,Balance of BOX ELDER,5222.0,1556.0,42583.0,4.0,5266.0,1581.0,46498.0,4.0,5311.0,1606.0,50503.0,4.0,5357.0,1631.0,54565.0,4.0,5589.0,1610.0,61740.0,4.0,5684.0,1675.0,63278.0,4.0,5753.0,1727.0,70980.0,4.0,5407.0,1759.0,72998.0,4.0,5547.0,1849.0,73002.0,4.0,5661.0,1888.0,73779.0,4.0,5295.0,1794.0,79839.0,4.0,5126.0,1705.0,85353.0,4,5328.0,1715.0,91893.0,4,5582.0,1778.0,102020.0,4,6109.0,1957.0,106143.0,4,6703.0,2065.0,113373.0,4
3,Bluffdale,6688.0,1499.0,110778.0,3.0,6567.0,1509.0,111804.0,4.0,6447.0,1532.0,112339.0,4.0,6326.0,1555.0,113010.0,4.0,6328.0,1578.0,113694.0,4.0,6374.0,1601.0,109131.0,4.0,10453.0,1932.0,99852.0,5.0,11411.0,2368.0,94218.0,5.0,12369.0,2805.0,93457.0,5.0,13328.0,3241.0,95597.0,5.0,14286.0,3677.0,99595.0,5.0,14802.0,3975.0,110985.0,5,16576.0,4596.0,106508.0,5,17460.0,5167.0,112658.0,5,18168.0,5483.0,125400.0,5,18797.0,5714.0,136480.0,5
4,Bountiful,43220.0,14440.0,58832.0,31.0,43178.0,14394.0,61396.0,31.0,43138.0,14345.0,64014.0,31.0,43094.0,14301.0,66674.0,31.0,42405.0,14096.0,74324.0,31.0,42782.0,14015.0,73252.0,31.0,42772.0,14042.0,73200.0,31.0,43257.0,14313.0,75155.0,31.0,43646.0,14351.0,78454.0,31.0,43591.0,14262.0,81501.0,31.0,43322.0,14324.0,85277.0,31.0,41527.0,13538.0,87430.0,31,43069.0,13782.0,93585.0,31,42676.0,13723.0,99351.0,31,42451.0,13747.0,103270.0,31,42334.0,13759.0,100449.0,31
5,Box Elder County North,1482.0,444.0,71921.0,1.0,1504.0,448.0,72934.0,1.0,1526.0,451.0,73946.0,1.0,1548.0,455.0,74958.0,1.0,1698.0,452.0,91250.0,1.0,1636.0,485.0,89135.0,1.0,1572.0,491.0,87917.0,1.0,1468.0,445.0,65625.0,1.0,1635.0,494.0,64167.0,1.0,1550.0,463.0,65797.0,1.0,1704.0,444.0,69722.0,1.0,1946.0,506.0,72083.0,1,1769.0,468.0,76346.0,1,1735.0,464.0,113611.0,1,1753.0,499.0,88250.0,1,1837.0,543.0,94554.0,1
6,Brigham City,13876.0,4849.0,36577.0,13.0,14054.0,4918.0,38257.0,13.0,14236.0,4986.0,39949.0,13.0,14414.0,5054.0,41667.0,13.0,14814.0,5186.0,49136.0,13.0,14763.0,5161.0,48418.0,13.0,14879.0,5055.0,47015.0,13.0,15142.0,5372.0,44853.0,13.0,15160.0,5374.0,48459.0,13.0,15339.0,5579.0,49513.0,13.0,15712.0,5638.0,50474.0,13.0,15886.0,5611.0,53237.0,13,15897.0,5603.0,56390.0,13,16264.0,5815.0,59026.0,13,16465.0,5803.0,63795.0,13,16625.0,5778.0,69037.0,13
7,Cedar Fort,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [32]:
# Aggregation spot check: manually verify weighted income and TOTHH for one city-year.
check_year = acs_years[0]
check_city = city_income_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]

check_df = bg_acs_by_year[check_year].copy()
check_df[HT_INCOME_CONFIG["bg_key"]] = check_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
check_df = check_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")
check_df = check_df[
    (check_df[HT_INCOME_CONFIG["city_key"]] == check_city)
    & check_df["median_income"].notna()
    & check_df["households"].notna()
    & (check_df["households"] > 0)
].copy()
check_df["weighted_income"] = check_df["median_income"] * check_df["households"]

computed_inc = round(check_df["weighted_income"].sum() / check_df["households"].sum(), 0)
computed_hh = check_df["households"].sum()
stored_inc = city_income_df.loc[
    city_income_df[HT_INCOME_CONFIG["city_key"]] == check_city, f"MEDINC{check_year}"
].iloc[0]
stored_hh = city_income_df.loc[
    city_income_df[HT_INCOME_CONFIG["city_key"]] == check_city, f"TOTHH{check_year}"
].iloc[0]

print(f"City: {check_city} | Year: {check_year}")
print(
    f"Manually computed MEDINC : {computed_inc}  |  Stored: {stored_inc}  |  Match: {computed_inc == stored_inc}"
)
print(f"Manually computed TOTHH  : {computed_hh}   |  Stored: {stored_hh}")


City: Alpine | Year: 2009
Manually computed MEDINC : 80435.0  |  Stored: 80435.0  |  Match: True
Manually computed TOTHH  : 2263.0   |  Stored: 2263.0


## 11. Fetch & Cache ACS Place Income / Build Hybrid MEDINC

For incorporated cities whose `CITYAREA` name matches a Census Place name exactly (~74
cities), ACS Place-level `B19013_001E` is more accurate than the BG-weighted estimate
because it is the directly reported Census figure rather than a weighted mean of
suppressed / interpolated block-group values.

**Source hierarchy applied to `MEDINC{YYYY}`:**

| City type | Source |
|---|---|
| Exact Census Place name match (~74 cities) | ACS Place B19013 |
| `Davis County` / `Weber County` whole-county TDM rows | ACS County (Section 7 CSV) |
| Sub-county / unincorporated / non-residential rows | BG-weighted (Section 10) |

`TOTPOP{YYYY}` and `TOTHH{YYYY}` always come from the BG-weighted aggregation.

**Cache file:** `.\Inputs\places_2020_acs_income.csv` — downloaded once, skipped on
re-runs.  Delete to force a full re-download (e.g. after adding a new target year).

In [33]:
from pygris import places as fetch_places

places_ref_path = HT_INCOME_CONFIG["places_ref_path"]

if os.path.exists(places_ref_path):
    place_income_filled_df = pd.read_csv(places_ref_path)
    print("Loaded cached Place ACS income:", places_ref_path)
    print("Shape:", place_income_filled_df.shape)

else:
    print("Fetching ACS Place-level income for all target years...")

    # Fetch 2020 Census Places geometry for Utah.
    places_ut_gdf = fetch_places(
        state=HT_INCOME_CONFIG["state"], year=2020, cache=HT_INCOME_CONFIG["pygris_cache"]
    )

    # ── Build a deduplicated GEOID → NAME lookup ──────────────────────────────
    # Utah has place names that belong to more than one GEOID
    # (e.g. "Enterprise" is both an incorporated city and a CDP).
    # When a NAME appears multiple times, keep the incorporated place
    # (NAMELSAD containing " city" or " town") and drop the CDP.
    # This prevents duplicate (place_name, year) rows from breaking the pivot.
    places_ref = places_ut_gdf[["GEOID", "NAME", "NAMELSAD"]].copy()
    places_ref["is_incorporated"] = (
        places_ref["NAMELSAD"].str.lower().str.contains(r"\bcity\b|\btown\b", regex=True)
    )

    # Sort so incorporated rows come first, then deduplicate on NAME
    places_dedup = places_ref.sort_values("is_incorporated", ascending=False).drop_duplicates(
        subset="NAME", keep="first"
    )

    dupe_names = places_ref.groupby("NAME")["GEOID"].nunique().loc[lambda s: s > 1].index.tolist()
    if dupe_names:
        print(f"Duplicate place names resolved (kept incorporated): {dupe_names}")

    geoid_to_place_name = places_dedup.set_index("GEOID")["NAME"].to_dict()

    place_raw_parts = []
    place_year_status = []

    for year in acs_years:
        try:
            raw = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=["B19013_001E"],
                year=year,
                params={
                    "for": "place:*",
                    "in": f"state:{HT_INCOME_CONFIG['state_fips']}",
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            raw = raw.rename(columns={"B19013_001E": "place_medinc"})
            raw["place_medinc"] = pd.to_numeric(raw["place_medinc"], errors="coerce")
            raw.loc[raw["place_medinc"] <= 0, "place_medinc"] = np.nan
            raw["place_name"] = raw["GEOID"].astype(str).map(geoid_to_place_name)
            raw["year"] = year
            raw["medinc_source"] = "acs_place"

            place_raw_parts.append(
                raw[["place_name", "year", "place_medinc", "medinc_source"]].dropna(
                    subset=["place_name"]
                )
            )
            place_year_status.append({"year": year, "status": "ok", "n": len(raw)})

        except Exception as exc:
            place_year_status.append({"year": year, "status": "error", "message": str(exc)})

    print(pd.DataFrame(place_year_status).to_string(index=False))

    if not place_raw_parts:
        raise RuntimeError(
            "No Place ACS data fetched — check CENSUS_API_KEY and target year range."
        )

    place_panel_df = pd.concat(place_raw_parts, ignore_index=True)

    # Final guard: drop any remaining duplicate (place_name, year) pairs
    dupes = place_panel_df.duplicated(subset=["place_name", "year"], keep=False)
    if dupes.any():
        print(f"WARNING: {dupes.sum()} duplicate (place_name, year) rows — keeping first.")
        place_panel_df = place_panel_df.drop_duplicates(subset=["place_name", "year"], keep="first")

    # Apply the same interp/extrap pipeline used for BG, tract, and county panels.
    place_income_filled_df = fill_acs_panel(
        place_panel_df,
        geoid_col="place_name",
        value_source_pairs=[("place_medinc", "medinc_source")],
    )

    place_income_filled_df.to_csv(places_ref_path, index=False)
    print("Saved:", places_ref_path)
    print("Shape:", place_income_filled_df.shape)


Fetching ACS Place-level income for all target years...
Using FIPS code '49' for input 'UT'
Duplicate place names resolved (kept incorporated): ['Enterprise']
 year status   n
 2009     ok 293
 2010     ok 326
 2011     ok 326
 2012     ok 326
 2013     ok 326
 2014     ok 325
 2015     ok 325
 2016     ok 326
 2017     ok 326
 2018     ok 327
 2019     ok 327
 2020     ok 333
 2021     ok 333
 2022     ok 333
 2023     ok 334
 2024     ok 334
Saved: .\Inputs\places_2020_acs_income.csv
Shape: (5189, 4)


In [34]:
# TDM rows that correspond to a whole-county ACS figure.
# Sub-county rows (Box Elder North/South, SL County East Cyns variants, etc.)
# are intentionally excluded — they fall back to BG-weighted (Section 10).
COUNTY_TDM_ROWS = {"Davis County": "49011", "Weber County": "49057"}

# ── Pivot Place income to wide format ─────────────────────────────────────────
# pivot_table with aggfunc="mean" guards against any duplicate (place_name, year)
# pairs that might survive the dedup step above.
# Use acs_years to include all years in the Place income panel.
place_wide = place_income_filled_df.pivot_table(
    index="place_name", columns="year", values="place_medinc", aggfunc="mean"
)
place_wide.columns = [f"place_medinc_{yr}" for yr in place_wide.columns]
place_wide = place_wide.reset_index()

# ── Build county lookup from the already-cached county panel (Section 7) ──────
# county_acs_filled_df is already in memory — no new download needed.
county_wide = county_acs_filled_df.pivot(
    index=HT_INCOME_CONFIG["county_key"], columns="year", values="median_income"
)
county_wide.columns = [f"county_medinc_{yr}" for yr in county_wide.columns]
county_wide = county_wide.reset_index()

county_tdm_rows = []
for tdm_name, fips in COUNTY_TDM_ROWS.items():
    row = county_wide[county_wide[HT_INCOME_CONFIG["county_key"]] == fips]
    if len(row) == 1:
        entry = {"CITYAREA": tdm_name}
        for yr in acs_years:  # use acs_years — all income years
            col = f"county_medinc_{yr}"
            entry[col] = row[col].iloc[0] if col in row.columns else np.nan
        county_tdm_rows.append(entry)
    else:
        print(f"WARNING: county FIPS {fips} not found in county ACS for '{tdm_name}'")

county_tdm_df = pd.DataFrame(county_tdm_rows) if county_tdm_rows else pd.DataFrame()

# ── Start from BG-weighted MEDINC as the base ─────────────────────────────────
medinc_hybrid_df = city_income_df.copy()

# Join Place income
medinc_hybrid_df = medinc_hybrid_df.merge(
    place_wide, left_on="CITYAREA", right_on="place_name", how="left"
)

# Join county income for county TDM rows
if not county_tdm_df.empty:
    medinc_hybrid_df = medinc_hybrid_df.merge(county_tdm_df, on="CITYAREA", how="left")
else:
    for yr in target_years:
        medinc_hybrid_df[f"county_medinc_{yr}"] = np.nan

# ── Apply override hierarchy per year ─────────────────────────────────────────
# Priority: ACS Place > ACS County > BG-weighted (already in MEDINC{yr})
# medinc_source_records tracks which source each city resolved to — internal
# only, not exported (would require MEDINC_SRC{YYYY} wide columns).
medinc_source_records = []

for yr in acs_years:  # apply override for every ACS year
    place_col_yr = f"place_medinc_{yr}"
    county_col_yr = f"county_medinc_{yr}"
    medinc_col = f"MEDINC{yr}"

    for idx, row in medinc_hybrid_df.iterrows():
        place_val = row.get(place_col_yr, np.nan)
        county_val = row.get(county_col_yr, np.nan)
        bg_val = row[medinc_col]

        if pd.notna(place_val) and place_val > 0:
            medinc_hybrid_df.at[idx, medinc_col] = place_val
            src = "acs_place"
        elif pd.notna(county_val) and county_val > 0:
            medinc_hybrid_df.at[idx, medinc_col] = county_val
            src = "acs_county"
        else:
            src = "bg_weighted" if pd.notna(bg_val) and bg_val > 0 else "none"

        if yr == acs_years[-1]:  # record source for latest ACS year
            medinc_source_records.append({"CITYAREA": row["CITYAREA"], "medinc_source": src})

# Drop temporary join columns — not part of the export schema
temp_cols = (
    [f"place_medinc_{yr}" for yr in acs_years]
    + [f"county_medinc_{yr}" for yr in acs_years]
    + ["place_name"]
)
medinc_hybrid_df = medinc_hybrid_df.drop(
    columns=[c for c in temp_cols if c in medinc_hybrid_df.columns]
)
# ── Rename MEDINC{yr} → MEDINC{yr}_CITY ──────────────────────────────────────
# The city-level hybrid income is retained for reference under the new naming scheme.
city_medinc_rename = {f"MEDINC{yr}": f"MEDINC{yr}_CITY" for yr in acs_years}
medinc_hybrid_df = medinc_hybrid_df.rename(columns=city_medinc_rename)

print("medinc_hybrid_df shape:", medinc_hybrid_df.shape)


medinc_hybrid_df shape: (90, 65)


In [35]:
# Internal source summary for most-recent year — not exported.
source_summary_df = (
    pd.DataFrame(medinc_source_records)
    .merge(medinc_hybrid_df[["CITYAREA", f"MEDINC{acs_years[-1]}_CITY"]], on="CITYAREA", how="left")
    .rename(columns={f"MEDINC{acs_years[-1]}_CITY": "MEDINC_final"})
    .sort_values("medinc_source")
)

counts = source_summary_df["medinc_source"].value_counts()
print(f"MEDINC source breakdown ({acs_years[-1]}):")
print(f"  acs_place   : {counts.get('acs_place', 0):3d} cities")
print(f"  acs_county  : {counts.get('acs_county', 0):3d} cities")
print(f"  bg_weighted : {counts.get('bg_weighted', 0):3d} cities")
print(f"  none (NaN)  : {counts.get('none', 0):3d} cities")
print()
print(source_summary_df.to_string(index=False))


MEDINC source breakdown (2024):
  acs_place   :  75 cities
  acs_county  :   2 cities
  bg_weighted :  13 cities
  none (NaN)  :   0 cities

                   CITYAREA medinc_source  MEDINC_final
               Weber County    acs_county       90005.0
               Davis County    acs_county      110884.0
                     Alpine     acs_place      168929.0
             Salt Lake City     acs_place       75090.0
                      Salem     acs_place      111117.0
                        Roy     acs_place       91282.0
                   Riverton     acs_place      126910.0
                  Riverdale     acs_place       67323.0
                      Provo     acs_place       64171.0
              Pleasant View     acs_place      129462.0
             Pleasant Grove     acs_place      101073.0
                 Plain City     acs_place      132766.0
                      Perry     acs_place      112639.0
                     Payson     acs_place       89905.0
                   

## 12. Compute H+T Income Share and Build Export

Fields computed for each city-year:

```
HTCOST{YYYY}      = HCOST{YYYY} + TCOST{YYYY}           (monthly $, H+T years only)
MEDINC{YYYY}_CITY = city-level hybrid income             (all ACS years 2009–latest)
MEDINC{YYYY}_CNTY = county ACS income (B19013_001E)      (all ACS years 2009–latest)
HPLUST{YYYY}     = (HTCOST{YYYY} × 12) / MEDINC{YYYY}_CNTY  (H+T years only, 0–1)
```

`MEDINC{yr}_CITY` and `MEDINC{yr}_CNTY` are exported for **all** `acs_years` (2009–latest H+T year).
`HCOST`, `TCOST`, `HTCOST`, `HPLUST`, `TOTPOP`, `TOTHH` are exported for **H+T years only** (`target_years`).

**fillna logic (split by column type):**
- `HCOST`, `TCOST`, `HTCOST` → filled with **0** (no TDM cost assigned is a real zero)
- `HPLUST` → stays **NaN** where MEDINC_CNTY is null/zero (renders as N/A in the dashboard,
  not "0% of income spent on H+T")
- `MEDINC{YYYY}_CITY`, `MEDINC{YYYY}_CNTY`, `TOTPOP`, `TOTHH` → filled with **0** (consistent, downstream can filter)

A guard drops previously derived columns before merging so this section is safe to re-run.

In [36]:
# Drop previously derived columns so re-runs do not accumulate duplicates.
derived_prefixes = ("MEDINC", "TOTPOP", "TOTHH", "bg_count_", "HTCOST")
existing_derived = [c for c in ht_df.columns if c.startswith(derived_prefixes)]
if existing_derived:
    ht_df = ht_df.drop(columns=existing_derived)

# Merge city-level ACS income / households into the H+T table.
# medinc_hybrid_df now carries MEDINC{yr}_CITY, TOTPOP{yr}, TOTHH{yr}.
ht_df = ht_df.merge(medinc_hybrid_df, on=HT_INCOME_CONFIG["city_key"], how="left")

# ── Join county-level income (ALL ACS years) ───────────────────────────────
# Pivot the already-cached county panel to wide format covering acs_years.
# medinc_hybrid_df already has MEDINC{yr}_CITY for all acs_years (merged above).
county_wide = county_acs_filled_df.pivot(
    index=HT_INCOME_CONFIG["county_key"], columns="year", values="median_income"
)
county_wide.columns = [f"MEDINC{yr}_CNTY" for yr in county_wide.columns]
county_wide = county_wide.reset_index()

ht_df = ht_df.merge(county_wide, on=HT_INCOME_CONFIG["county_key"], how="left")

# Compute HTCOST and HPLUST for H+T years only (target_years).
# MEDINC{yr}_CITY and MEDINC{yr}_CNTY are already joined for all acs_years.
for year in target_years:
    htcost_col = f"HTCOST{year}"
    cnty_income_col = f"MEDINC{year}_CNTY"
    ht_df[htcost_col] = np.where(
        ht_df[f"HCOST{year}"].notna() & ht_df[f"TCOST{year}"].notna(),
        ht_df[f"HCOST{year}"] + ht_df[f"TCOST{year}"],
        np.nan,
    )
    ht_df[f"HPLUST{year}"] = np.where(
        ht_df[cnty_income_col].notna() & (ht_df[cnty_income_col] > 0),
        (ht_df[htcost_col] * 12) / ht_df[cnty_income_col],
        np.nan,  # stays NaN — not 0 — for no-income cities
    )

# Summary (show first 2 ACS years and first 2 H+T years).
share_cols = [f"HPLUST{year}" for year in target_years]
htcost_cols = [f"HTCOST{year}" for year in target_years]
city_income_cols = [f"MEDINC{year}_CITY" for year in acs_years]
cnty_income_cols = [f"MEDINC{year}_CNTY" for year in acs_years]

print(
    ht_df[
        [HT_INCOME_CONFIG["city_key"]]
        + city_income_cols[:2]
        + cnty_income_cols[:2]
        + htcost_cols[:2]
        + share_cols[:2]
    ].head()
)
print()
print("H+T income-share range by year:")
for year in target_years:
    s = ht_df[f"HPLUST{year}"]
    print(
        f"  {year}  non-null: {s.notna().sum():3d}"
        f"  null (no MEDINC_CNTY): {s.isna().sum():3d}"
        f"  min: {s.min():.4f}  max: {s.max():.4f}"
    )
print(
    f"\nACS income columns: {len(city_income_cols)} MEDINC_CITY + {len(cnty_income_cols)} MEDINC_CNTY ({acs_years[0]}–{acs_years[-1]})"
)


               CITYAREA  MEDINC2009_CITY  MEDINC2010_CITY  MEDINC2009_CNTY  \
0                Alpine         104436.0         107773.0          56752.0   
1                  Alta              NaN              NaN          56948.0   
2         American Fork          67124.0          69167.0          56752.0   
3  Balance of BOX ELDER          42583.0          46498.0          54670.0   
4              Benjamin              NaN              NaN          56752.0   

   MEDINC2010_CNTY  HTCOST2019  HTCOST2020  HPLUST2019  HPLUST2020  
0          56927.0      3036.0      3003.0    0.487939    0.467654  
1          58004.0       603.0       519.0    0.096654    0.080749  
2          56927.0      2315.0      2302.0    0.372062    0.358488  
3          55135.0      1018.0       857.0    0.196295    0.161767  
4          56927.0       694.0       643.0    0.111538    0.100134  

H+T income-share range by year:
  2019  non-null: 101  null (no MEDINC_CNTY):   0  min: 0.0810  max: 0.5116
  2020  

In [37]:
# Spot check: manually verify one city-year end-to-end.
check_year = target_years[0]
check_city = ht_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]
row = ht_df.loc[ht_df[HT_INCOME_CONFIG["city_key"]] == check_city].iloc[0]

print(f"City             : {check_city}")
print(f"Year             : {check_year}")
print(f"HCOST (mo)       : {row[f'HCOST{check_year}']}")
print(f"TCOST (mo)       : {row[f'TCOST{check_year}']}")
print(f"HTCOST (mo)      : {row[f'HTCOST{check_year}']}  (= HCOST + TCOST)")
print(f"MEDINC_CITY      : {row[f'MEDINC{check_year}_CITY']}")
print(f"MEDINC_CNTY      : {row[f'MEDINC{check_year}_CNTY']}")
print(f"TOTHH            : {row[f'TOTHH{check_year}']}")
print(
    f"Income share     : {row[f'HPLUST{check_year}']:.4f}  (= HTCOST × 12 / MEDINC_CNTY, expect 0-1)"
)


City             : Alpine
Year             : 2019
HCOST (mo)       : 2255
TCOST (mo)       : 781
HTCOST (mo)      : 3036.0  (= HCOST + TCOST)
MEDINC_CITY      : 129239.0
MEDINC_CNTY      : 74665.0
TOTHH            : 2567.0
Income share     : 0.4879  (= HTCOST × 12 / MEDINC_CNTY, expect 0-1)


In [38]:
# Build the export DataFrame: city geometry + H+T data.
export_df = city_area_shp[[HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]].merge(
    ht_df, on=HT_INCOME_CONFIG["city_key"], how="left"
)


export_df["SUBAREA"] = export_df["SUBAREA"].replace(HT_INCOME_CONFIG["workshop_area_lookup"])

print("Pre-collapse rows:", len(export_df))
print("Pre-collapse cols:", len(export_df.columns))


Pre-collapse rows: 109
Pre-collapse cols: 110


In [39]:
# Drop intermediate columns not part of the final schema.
drop_prefixes = ("bg_count_",)
drop_cols = [c for c in export_df.columns if c.startswith(drop_prefixes)]
if drop_cols:
    export_df = export_df.drop(columns=drop_cols)

# ── fillna — split by column type ────────────────────────────────────────────
# HCOST, TCOST, HTCOST: fill NaN with 0 (no TDM cost = real zero).
cost_fill_cols = []
for year in target_years:
    cost_fill_cols.extend([f"HCOST{year}", f"TCOST{year}", f"HTCOST{year}"])
cost_fill_cols = [c for c in cost_fill_cols if c in export_df.columns]
export_df[cost_fill_cols] = export_df[cost_fill_cols].fillna(0)

# MEDINC{yr}_CITY, MEDINC{yr}_CNTY, TOTPOP, TOTHH: fill NaN with 0 (non-residential / unmatched zones).
stat_fill_cols = [
    c for c in export_df.columns if any(c.startswith(p) for p in ("MEDINC", "TOTPOP", "TOTHH"))
]
export_df[stat_fill_cols] = export_df[stat_fill_cols].fillna(0)

# HPLUST: intentionally left as NaN where MEDINC was null/zero.
# The dashboard renders NaN as N/A rather than "0% of income spent on H+T".

# ── Column ordering ────────────────────────────────────────────────────────────
# Cost/share columns: H+T years only (target_years).
# Income columns: all ACS years (acs_years).
ordered_cols = [HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]
for prefix in ["HCOST", "TCOST", "HTCOST", "HPLUST", "TOTPOP", "TOTHH"]:
    for year in target_years:  # H+T years only
        col = f"{prefix}{year}"
        if col in export_df.columns:
            ordered_cols.append(col)
# MEDINC covers ALL acs_years (2009 – latest H+T year).
for year in acs_years:
    for col in [f"MEDINC{year}_CITY", f"MEDINC{year}_CNTY"]:
        if col in export_df.columns:
            ordered_cols.append(col)
remaining_cols = [c for c in export_df.columns if c not in ordered_cols]
export_df = export_df[ordered_cols + remaining_cols]

print("Final export columns:")
print(export_df.columns.tolist())
export_df.head()


Final export columns:
['CITYAREA', 'SUBAREA', 'CO_NAME', 'SHAPE', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024', 'TOTHH2019', 'TOTHH2020', 'TOTHH2021', 'TOTHH2022', 'TOTHH2023', 'TOTHH2024', 'MEDINC2009_CITY', 'MEDINC2009_CNTY', 'MEDINC2010_CITY', 'MEDINC2010_CNTY', 'MEDINC2011_CITY', 'MEDINC2011_CNTY', 'MEDINC2012_CITY', 'MEDINC2012_CNTY', 'MEDINC2013_CITY', 'MEDINC2013_CNTY', 'MEDINC2014_CITY', 'MEDINC2014_CNTY', 'MEDINC2015_CITY', 'MEDINC2015_CNTY', 'MEDINC2016_CITY', 'MEDINC2016_CNTY', 'MEDINC2017_CITY', 'MEDINC2017_CNTY', 'MEDINC2018_CITY', 'MEDINC2018_CNTY', 'MEDINC2019_CITY', 'MEDINC2019_CNTY', 'MEDINC2020_CITY', 'MEDINC2020

,CITYAREA,SUBAREA,CO_NAME,SHAPE,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HTCOST2019,HTCOST2020,HTCOST2021,HTCOST2022,HTCOST2023,HTCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,TOTPOP2019,TOTPOP2020,TOTPOP2021,TOTPOP2022,TOTPOP2023,TOTPOP2024,TOTHH2019,TOTHH2020,TOTHH2021,TOTHH2022,TOTHH2023,TOTHH2024,MEDINC2009_CITY,MEDINC2009_CNTY,MEDINC2010_CITY,MEDINC2010_CNTY,MEDINC2011_CITY,MEDINC2011_CNTY,MEDINC2012_CITY,MEDINC2012_CNTY,MEDINC2013_CITY,MEDINC2013_CNTY,MEDINC2014_CITY,MEDINC2014_CNTY,MEDINC2015_CITY,MEDINC2015_CNTY,MEDINC2016_CITY,MEDINC2016_CNTY,MEDINC2017_CITY,MEDINC2017_CNTY,MEDINC2018_CITY,MEDINC2018_CNTY,MEDINC2019_CITY,MEDINC2019_CNTY,MEDINC2020_CITY,MEDINC2020_CNTY,MEDINC2021_CITY,MEDINC2021_CNTY,MEDINC2022_CITY,MEDINC2022_CNTY,MEDINC2023_CITY,MEDINC2023_CNTY,MEDINC2024_CITY,MEDINC2024_CNTY,COUNTY_NAME,county_geoid,TOTPOP2009,TOTHH2009,TOTPOP2010,TOTHH2010,TOTPOP2011,TOTHH2011,TOTPOP2012,TOTHH2012,TOTPOP2013,TOTHH2013,TOTPOP2014,TOTHH2014,TOTPOP2015,TOTHH2015,TOTPOP2016,TOTHH2016,TOTPOP2017,TOTHH2017,TOTPOP2018,TOTHH2018
0,Alpine,Utah County North,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",2255.0,2299.0,2741.0,3914.0,4274.0,4341.0,781.0,704.0,751.0,758.0,769.0,775.0,3036.0,3003.0,3492.0,4672.0,5043.0,5116.0,0.487939,0.467654,0.505519,0.614312,0.624668,0.609828,9794.0,10208.0,9756.0,9484.0,9649.0,9566.0,2567.0,2627.0,2552.0,2554.0,2697.0,2796.0,104436.0,56752.0,107773.0,56927.0,98785.0,59338.0,97449.0,59864.0,92443.0,60196.0,98589.0,60830.0,92305.0,62180.0,102122.0,64321.0,112727.0,67042.0,121667.0,70408.0,129239.0,74665.0,123450.0,77057.0,138438.0,82893.0,161602.0,91263.0,156786.0,96877.0,168929.0,100671.0,Utah,49049,9313.0,2263.0,9349.0,2291.0,9381.0,2320.0,9415.0,2352.0,9409.0,2365.0,9159.0,2396.0,9250.0,2516.0,9686.0,2468.0,9692.0,2472.0,9924.0,2557.0
1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",0.0,0.0,0.0,0.0,0.0,0.0,603.0,519.0,555.0,575.0,588.0,572.0,603.0,519.0,555.0,575.0,588.0,572.0,0.096654,0.080749,0.081016,0.076657,0.074542,0.070404,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,56948.0,0.0,58004.0,0.0,59168.0,0.0,59626.0,0.0,60555.0,0.0,61446.0,0.0,62117.0,0.0,64601.0,0.0,67922.0,0.0,71230.0,0.0,74865.0,0.0,77128.0,0.0,82206.0,0.0,90011.0,0.0,94658.0,0.0,97494.0,Salt Lake,49035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,American Fork,Utah County North,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",1700.0,1726.0,2045.0,2767.0,3073.0,3366.0,615.0,576.0,597.0,603.0,609.0,613.0,2315.0,2302.0,2642.0,3370.0,3682.0,3979.0,0.372062,0.358488,0.382469,0.443115,0.456083,0.474297,33624.0,33780.0,34725.0,36227.0,37988.0,39785.0,9733.0,9799.0,10334.0,10841.0,11642.0,12326.0,67124.0,56752.0,69167.0,56927.0,68725.0,59338.0,66504.0,59864.0,67595.0,60196.0,66687.0,60830.0,65101.0,62180.0,69493.0,64321.0,70926.0,67042.0,74192.0,70408.0,77857.0,74665.0,78690.0,77057.0,82772.0,82893.0,90490.0,91263.0,95823.0,96877.0,98878.0,100671.0,Utah,49049,27246.0,7171.0,27456.0,7307.0,27537.0,7451.0,27614.0,7595.0,26935.0,7707.0,27518.0,7856.0,28153.0,8101.0,28621.0,8049.0,29599.0,8426.0,31965.0,8970.0
3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",0.0,0.0,0.0,0.0,0.0,0.0,1018.0,857.0,907.0,970.0,1070.0,970.0,1018.0,857.0,907.0,970.0,1070.0,970.0,0.196295,0.161767,0.161278,0.159958,0.164901,0.137670,5295.0,5126.0,5328.0,5582.0,6109.0,6703.0,1794.0,1705.0,1715.0,1778.0,1957.0,2065.0,42583.0,54670.0,46498.0,55135.0,50503.0,55588.0,54565.0,55918.0,61740.0,57292.0,63278.0,57336.0,70980.0,55038.0,72998.0,55514.0,73002.0,58835.0,73779.0,59937.0,79839.0,62233.0,85353.0,63573.0,91893.0,67486.0,102020.0,72769.0,106143.0,77865.0,113373.0,84550.0,Box Elder,49003,5222.0,1556.0,5266.0,1581.0,5311.0,1606.0,5357.0,1631.0,5589.0,1610.0,5684.0,1675.0,5753.0,1727.0,5407.0,1759.0,5547.0,1849.0,5661.0,1888

In [40]:
# toggle: comment out to preserve NaN/null in export
export_df = export_df.fillna({c: 0 for c in export_df.columns if c != "SHAPE"})


In [41]:
export_fc = os.path.join(gdb2, "Affordability_Housing_Transportation_Costs")

if arcpy.Exists(export_fc):
    arcpy.management.Delete(export_fc)

export_df.spatial.to_featureclass(location=export_fc, sanitize_columns=False)
print("Exported:", export_fc)

export_df.drop(columns=["SHAPE"], errors="ignore").to_csv(
    os.path.join(outputs[0], "Affordability_Housing_Transportation_Costs.csv"), index=False
)

print("CSV exported:", os.path.join(outputs[0], "Affordability_Housing_Transportation_Costs.csv"))


Exported: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs
CSV exported: .\Outputs\Affordability_Housing_Transportation_Costs.csv


## 13. Validate Export

In [42]:
export_check = pd.DataFrame.spatial.from_featureclass(export_fc)

print("Export shape:", export_check.shape)
print(export_check.columns.tolist())

# Cost/share spot-check uses earliest H+T year; income uses earliest ACS year (2009).
check_cols = [
    "CITYAREA",
    "SUBAREA",
    "CO_NAME",
    f"HCOST{target_years[0]}",
    f"TCOST{target_years[0]}",
    f"HTCOST{target_years[0]}",
    f"HPLUST{target_years[0]}",
    f"MEDINC{acs_years[0]}_CITY",  # earliest ACS year (2009)
    f"MEDINC{acs_years[0]}_CNTY",
    f"MEDINC{target_years[0]}_CITY",  # earliest H+T year
    f"MEDINC{target_years[0]}_CNTY",
    f"TOTPOP{target_years[0]}",
    f"TOTHH{target_years[0]}",
]
print(export_check[check_cols].head())


Export shape: (109, 95)
['OBJECTID', 'CITYAREA', 'SUBAREA', 'CO_NAME', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024', 'TOTHH2019', 'TOTHH2020', 'TOTHH2021', 'TOTHH2022', 'TOTHH2023', 'TOTHH2024', 'MEDINC2009_CITY', 'MEDINC2009_CNTY', 'MEDINC2010_CITY', 'MEDINC2010_CNTY', 'MEDINC2011_CITY', 'MEDINC2011_CNTY', 'MEDINC2012_CITY', 'MEDINC2012_CNTY', 'MEDINC2013_CITY', 'MEDINC2013_CNTY', 'MEDINC2014_CITY', 'MEDINC2014_CNTY', 'MEDINC2015_CITY', 'MEDINC2015_CNTY', 'MEDINC2016_CITY', 'MEDINC2016_CNTY', 'MEDINC2017_CITY', 'MEDINC2017_CNTY', 'MEDINC2018_CITY', 'MEDINC2018_CNTY', 'MEDINC2019_CITY', 'MEDINC2019_CNTY', 'MEDINC2020_CITY', 'MEDIN

In [43]:
# Per-year range check on the income share (stored in HPLUST columns).
# Expected: values between 0.0 and ~0.6; max above 1.0 would indicate a calculation error.
# NaN count expected for non-residential TDM zones (Utah Lake, Camp Williams, etc.).
print("Export feature class:", export_fc)
print("Rows   :", len(export_check))
print("Columns:", len(export_check.columns))
print()
print("H+T income-share validation (H+T years only):")
for year in target_years:
    s = export_check[f"HPLUST{year}"]
    sc = export_check[f"MEDINC{year}_CNTY"]
    print(
        f"  {year}  HPLUST non-null: {s.notna().sum():3d}"
        f"  null (N/A): {s.isna().sum():3d}"
        f"  min: {s.min():.4f}  max: {s.max():.4f}"
        f"  |  MEDINC_CNTY min: {sc.min():.0f}  max: {sc.max():.0f}"
    )

print(f"\nACS income columns present for all acs_years ({acs_years[0]}–{acs_years[-1]}):")
for year in acs_years:
    city_col = f"MEDINC{year}_CITY"
    cnty_col = f"MEDINC{year}_CNTY"
    c_ok = city_col in export_check.columns
    k_ok = cnty_col in export_check.columns
    print(
        f"  {year}  MEDINC_CITY: {'OK' if c_ok else 'MISSING':7s}"
        f"  MEDINC_CNTY: {'OK' if k_ok else 'MISSING'}"
    )


Export feature class: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs
Rows   : 109
Columns: 95

H+T income-share validation (H+T years only):
  2019  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.5116  |  MEDINC_CNTY min: 0  max: 83310
  2020  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.4955  |  MEDINC_CNTY min: 0  max: 87570
  2021  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.5817  |  MEDINC_CNTY min: 0  max: 92765
  2022  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.6196  |  MEDINC_CNTY min: 0  max: 101285
  2023  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.6506  |  MEDINC_CNTY min: 0  max: 108058
  2024  HPLUST non-null: 109  null (N/A):   0  min: 0.0000  max: 0.6280  |  MEDINC_CNTY min: 0  max: 110884

ACS income columns present for all acs_years (2009–2024):
  2009  MEDINC_CITY: OK       MEDINC_CNTY: OK
  2010  MEDINC_CITY: OK       MEDINC_CNTY: OK
  2011  ME

In [44]:
# Spot check: Alpine in the most recent year.
check_city = "Alpine"
check_ht_year = target_years[-1]  # latest H+T year for cost/share columns
check_acs_year = acs_years[0]  # earliest ACS year (2009) to verify extended income range

print(
    f"Spot check: {check_city}  |  H+T year: {check_ht_year}  |  ACS income year: {check_acs_year}"
)
export_check.loc[
    export_check["CITYAREA"] == check_city,
    [
        "CITYAREA",
        "CO_NAME",
        f"HCOST{check_ht_year}",
        f"TCOST{check_ht_year}",
        f"HTCOST{check_ht_year}",
        f"HPLUST{check_ht_year}",
        f"MEDINC{check_acs_year}_CITY",  # 2009 income
        f"MEDINC{check_acs_year}_CNTY",
        f"MEDINC{check_ht_year}_CITY",  # latest H+T year income
        f"MEDINC{check_ht_year}_CNTY",
        f"TOTHH{check_ht_year}",
    ],
]


Spot check: Alpine  |  H+T year: 2024  |  ACS income year: 2009


,CITYAREA,CO_NAME,HCOST2024,TCOST2024,HTCOST2024,HPLUST2024,MEDINC2009_CITY,MEDINC2009_CNTY,MEDINC2024_CITY,MEDINC2024_CNTY,TOTHH2024
0,Alpine,UTAH,4341,775,5116,0.609828,104436,56752,168929,100671,2796
